In [3]:
# ── Strategic Reasoning Layer (replaces translate_concept_phrase + build_reasoning)
#
# Philosophy: Observation → Strategic implication → Tactical decision.
# Each concept maps to (observation, implication) pairs at different intensities.
# The composer prioritizes by strategic urgency, varies sentence openings,
# and closes with a tactical verdict.

import hashlib

# Strategic urgency ranking — how time-critical each concept is when it fires.
# Higher = more tactically urgent (drives the decision harder in the narrative).
CONCEPT_URGENCY = {
    0: 2,   # degradation_severity — gradual, gives a window
    1: 2,   # pace_decay_rate — gradual
    2: 3,   # strategic_window — timing-sensitive
    3: 2,   # track_position_risk — situational
    4: 4,   # undercut_pressure — time-critical, respond now
    5: 5,   # endgame_proximity — overrides everything when high
}


def concept_clause(concept_idx, score):
    """Return (observation, strategic_implication) for a concept at a given score.
    Observation = what is happening. Implication = why it matters for the race.
    Returns None if the concept is not meaningfully active.
    """
    if score < 0.2:
        return None
    
    if concept_idx == 0:   # degradation_severity
        if score >= 0.7:
            return ("the tyres are deep into their wear phase",
                    "another few laps at racing pace risks a sharp drop-off and lost lap time")
        elif score >= 0.4:
            return ("the tyres are wearing meaningfully",
                    "the performance window is starting to close")
        else:
            return ("the tyres are picking up early wear",
                    "degradation is worth monitoring but not yet critical")
    
    elif concept_idx == 1:   # pace_decay_rate
        if score >= 0.5:
            return ("lap times are falling away quickly",
                    "the car is bleeding time to rivals every lap it stays out")
        else:
            return ("lap pace is beginning to slip",
                    "the trend is downward but still manageable")
    
    elif concept_idx == 2:   # strategic_window
        if score >= 0.7:
            return ("the optimal pit window is open right now",
                    "stopping now maximizes the tyre-offset advantage for the rest of the race")
        elif score >= 0.4:
            return ("the pit window is opening",
                    "the next few laps are the natural moment to commit to a stop")
        else:
            return ("we are near the edge of the pit window",
                    "timing is becoming a factor in the decision")
    
    elif concept_idx == 3:   # track_position_risk
        if score >= 0.6:
            return ("a hard-won track position is on the line",
                    "pitting now would likely surrender places that are difficult to win back on track")
        elif score >= 0.3:
            return ("there is track position worth protecting",
                    "any stop needs to weigh the cost of rejoining in traffic")
        else:
            return None
    
    elif concept_idx == 4:   # undercut_pressure
        if score >= 0.7:
            return ("a rival has pitted and is poised to undercut",
                    "failing to respond this lap likely hands them the position once stops cycle through")
        elif score >= 0.4:
            return ("competitors in the immediate fight are beginning to move",
                    "the undercut threat is building and the window to cover it is narrowing")
        else:
            return ("there is early movement in the local battle",
                    "worth watching in case it forces a reactive stop")
    
    elif concept_idx == 5:   # endgame_proximity
        if score >= 0.7:
            return ("the race is in its final phase",
                    "there is no longer enough distance left to recover the time a pit stop costs")
        elif score >= 0.4:
            return ("the race is approaching its closing stage",
                    "the payback window for a fresh set of tyres is shrinking fast")
        else:
            return ("the closing laps are coming into view",
                    "the cost-benefit of a late stop is tightening")


# Varied sentence openers, selected deterministically per-row to avoid repetition.
PIT_OPENERS = [
    "{obs}, and {imp} — the call is to pit.",
    "With {obs}, {imp}. Bringing the car in is the right move.",
    "{obs}. {imp_cap}, so a stop is justified now.",
    "The read here: {obs}. {imp_cap} — pitting is the stronger play.",
]
STAY_OPENERS = [
    "{obs}, but {imp} — better to stay out.",
    "Although {obs}, {imp}. Holding track position is the right move.",
    "{obs}. {imp_cap}, so staying out is the wiser option for now.",
    "The read here: {obs}, yet {imp} — the car stays out.",
]
SECONDARY_CONNECTORS = [
    "On top of that, {obs2} — {imp2}.",
    "Reinforcing the call, {obs2}, which means {imp2}.",
    "There is a second factor too: {obs2}, and {imp2}.",
]


def _pick(options, seed_int):
    """Deterministic choice from a list based on an integer seed."""
    return options[seed_int % len(options)]

def build_strategic_reasoning(concepts, contributions, decision, row_id, raw_row=None):
    """Compose strategist-style reasoning with coherence guarding.
    
    If raw_row is provided, concepts are coherence-filtered before narration.
    """
    suppressed = []
    if raw_row is not None:
        concepts, suppressed = coherence_filter(concepts, raw_row)
    
    # Gather active concepts with their clauses
    active = []
    for j in range(6):
        clause = concept_clause(j, concepts[j])
        if clause is not None:
            priority = CONCEPT_URGENCY[j] * abs(contributions[j])
            active.append((priority, j, clause))
    
    # Fallback if nothing is active (either genuinely quiet, or all suppressed)
    if len(active) == 0:
        if 'ALL (non-race event)' in suppressed:
            return "This is a non-race session, so pit-strategy reasoning does not apply."
        if decision == "PIT":
            return ("No single factor dominates, but the combined picture tips "
                    "just far enough toward a stop to justify pitting.")
        else:
            return ("Nothing is forcing a decision — the situation is stable, "
                    "so the car holds station and stays out.")
    
    active.sort(reverse=True, key=lambda t: t[0])
    seed_int = int(hashlib.md5(str(int(row_id)).encode()).hexdigest(), 16)
    
    # Primary concept
    _, j1, (obs1, imp1) = active[0]
    openers = PIT_OPENERS if decision == "PIT" else STAY_OPENERS
    opener = _pick(openers, seed_int)
    sentence = opener.format(obs=obs1, imp=imp1, imp_cap=imp1[0].upper() + imp1[1:])
    
    # Secondary concept
    if len(active) >= 2:
        priority2, j2, (obs2, imp2) = active[1]
        if priority2 >= 0.3:
            connector = _pick(SECONDARY_CONNECTORS, seed_int // 7)
            sentence += " " + connector.format(obs2=obs2, imp2=imp2)
    
    return sentence

In [4]:
# Notebook 08 — Inference on Kaggle Test Set with Per-Row Reasoning
#
# Uses the joint λ=20 model (public AUC 0.909, 5/6 concepts preserved).
# Outputs a CSV with: id, decision, pit_prob, 6 concept scores, reasoning.

from pathlib import Path
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Paths
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

X_TEST_PICKLE = DATA_DIR / 'x_test.pkl'
KAGGLE_TEST_CSV = DATA_DIR / 'test.csv'
JOINT_CHECKPOINT = CHECKPOINTS_DIR / 'joint_model_lambda20.pt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INPUT_DIM = 13
CONCEPT_DIM = 6

FEATURE_NAMES = [
    'lap_time_norm', 'lap_time_delta', 'cumulative_degradation',
    'tyre_life_norm', 'compound_soft', 'compound_medium', 'compound_hard',
    'compound_inter', 'compound_wet', 'position_norm', 'position_change_norm',
    'race_progress', 'stint_norm',
]
CONCEPT_NAMES = [
    'degradation_severity', 'pace_decay_rate', 'strategic_window',
    'track_position_risk', 'undercut_pressure', 'endgame_proximity',
]


# ── Model classes (must match what was trained)
class JointConceptBlock(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, concept_dim=CONCEPT_DIM, hidden_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, concept_dim)
    def forward(self, x):
        h = F.relu(self.ln1(self.fc1(x)))
        return torch.sigmoid(self.fc2(h))


class JointDecisionBlock(nn.Module):
    def __init__(self, concept_dim=CONCEPT_DIM):
        super().__init__()
        self.linear = nn.Linear(concept_dim, 1)
    def forward(self, c):
        return self.linear(c).squeeze(-1)


class JointModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.concept_block = JointConceptBlock()
        self.decision_block = JointDecisionBlock()
    def forward(self, x):
        c = self.concept_block(x)
        logit = self.decision_block(c)
        return logit, c


# # ── Plain-English phrase library (no concept names exposed in output text)
# def translate_concept_phrase(concept_idx, score):
#     if concept_idx == 0:   # degradation_severity
#         if score >= 0.7: return "the tyres are heavily worn"
#         elif score >= 0.4: return "the tyres are showing meaningful wear"
#         elif score >= 0.2: return "the tyres are starting to wear"
#         else: return "the tyres are still in good shape"
#     elif concept_idx == 1:   # pace_decay_rate
#         if score >= 0.5: return "lap pace is dropping fast"
#         elif score >= 0.2: return "lap pace is starting to slip"
#         else: return "lap pace is holding steady"
#     elif concept_idx == 2:   # strategic_window
#         if score >= 0.7: return "we are right in the typical pit window"
#         elif score >= 0.4: return "we are entering the pit window"
#         elif score >= 0.2: return "we are near the edge of the pit window"
#         else: return "we are outside the typical pit window"
#     elif concept_idx == 3:   # track_position_risk
#         if score >= 0.6: return "we would lose a strong position by pitting"
#         elif score >= 0.3: return "track position is worth protecting"
#         else: return "track position is not at risk"
#     elif concept_idx == 4:   # undercut_pressure
#         if score >= 0.7: return "a rival just pitted and is threatening an undercut"
#         elif score >= 0.4: return "competitors in the local battle are starting to move"
#         elif score >= 0.2: return "there is some early sign of competitor activity"
#         else: return "the local battle is quiet"
#     elif concept_idx == 5:   # endgame_proximity
#         if score >= 0.7: return "the race is nearly over"
#         elif score >= 0.4: return "we are approaching the end of the race"
#         elif score >= 0.2: return "race end is on the horizon"
#         else: return "there is plenty of race left"


# def build_reasoning(concepts, contributions, decision):
#     """Compose plain-English reasoning from the top driving concepts.
#     Uses up to 2 active concepts (score >= 0.2) ranked by |contribution|.
#     """
#     # Find active concepts ranked by absolute contribution magnitude
#     active_mask = concepts >= 0.2
#     active_idx = np.where(active_mask)[0]
    
#     if len(active_idx) == 0:
#         # No active concepts — minimal generic statement
#         return ("Multiple soft signals together justify a pit stop now."
#                 if decision == "PIT"
#                 else "The current state is stable enough to stay out.")
    
#     # Rank active concepts by absolute contribution
#     sorted_idx = active_idx[np.argsort(np.abs(contributions[active_idx]))[::-1]]
#     top = sorted_idx[:2] if len(sorted_idx) >= 2 else sorted_idx[:1]
#     phrases = [translate_concept_phrase(j, concepts[j]) for j in top]
    
#     if decision == "PIT":
#         if len(phrases) == 1:
#             return f"Because {phrases[0]}, a pit stop is the right call now."
#         else:
#             return f"Because {phrases[0]} and {phrases[1]}, a pit stop is the right call now."
#     else:
#         if len(phrases) == 1:
#             return f"Because {phrases[0]}, staying out is the better option for now."
#         else:
#             return f"Because {phrases[0]} and {phrases[1]}, staying out is the better option for now."


# ── Load model
print("=" * 70)
print("LOADING JOINT λ=20 MODEL")
print("=" * 70)

ckpt = torch.load(JOINT_CHECKPOINT, weights_only=False)
model = JointModel().to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Extract decision block weights and bias for contribution computation
weights = model.decision_block.linear.weight[0].detach().cpu().numpy()
bias = model.decision_block.linear.bias.item()

print(f"\n  Lambda used:      {ckpt.get('lambda_concept', 'unknown')}")
print(f"  Best val AUC:     {ckpt.get('best_val_auc', 'unknown')}")
print(f"  Total params:     {sum(p.numel() for p in model.parameters()):,}")
print(f"\n  Decision block weights:")
for j, name in enumerate(CONCEPT_NAMES):
    print(f"    {name:<24s}  {weights[j]:+.4f}")
print(f"    {'bias':<24s}  {bias:+.4f}")


# ── Load test data (both pickled features and raw CSV for context)
print("\n" + "=" * 70)
print("LOADING KAGGLE TEST DATA")
print("=" * 70)

x_test_df = pd.read_pickle(X_TEST_PICKLE)
x_cols = [f'x_{n}' for n in FEATURE_NAMES]
x_test_arr = x_test_df[x_cols].values.astype(np.float32)
test_ids = x_test_df['id'].values

# Raw test CSV for human-readable context (driver, race, lap, etc.)
kaggle_test_raw = pd.read_csv(KAGGLE_TEST_CSV)
kaggle_test_raw.columns = kaggle_test_raw.columns.str.strip()
assert (kaggle_test_raw['id'].values == test_ids).all(), "id mismatch between pickle and CSV"

print(f"  Test rows: {len(x_test_arr):,}")


# ── Run inference (chunked for memory efficiency)
print("\n" + "=" * 70)
print("RUNNING INFERENCE")
print("=" * 70)

t_start = time.time()
chunk_size = 50000
all_concepts, all_probs = [], []

with torch.no_grad():
    x_tensor = torch.from_numpy(x_test_arr).to(DEVICE)
    for i in range(0, len(x_tensor), chunk_size):
        chunk = x_tensor[i:i+chunk_size]
        logit, c = model(chunk)
        all_probs.append(torch.sigmoid(logit).cpu().numpy())
        all_concepts.append(c.cpu().numpy())
        print(f"  Processed {min(i+chunk_size, len(x_tensor)):,} / {len(x_tensor):,}")

test_probs = np.concatenate(all_probs, axis=0)
test_concepts = np.concatenate(all_concepts, axis=0)

print(f"\n  Inference complete in {time.time() - t_start:.1f}s")


# ── Build the rich output table
print("\n" + "=" * 70)
print("BUILDING OUTPUT TABLE WITH REASONING")
print("=" * 70)

t_start = time.time()

# Compute contributions for all rows: shape (188165, 6)
contributions_all = test_concepts * weights

# Decisions
decisions = np.where(test_probs >= 0.5, "PIT", "NO_PIT")

# Generate reasoning per row (this is the slow part — pure Python loop)
print(f"  Generating reasoning for {len(test_probs):,} rows...")
reasoning_list = []
for i in range(len(test_probs)):
    # OLD:
    #r = build_reasoning(test_concepts[i], contributions_all[i], decisions[i])
    
    # NEW:
    r = build_strategic_reasoning(test_concepts[i], contributions_all[i], decisions[i], test_ids[i])
    reasoning_list.append(r)
    if (i + 1) % 20000 == 0:
        print(f"    {i+1:,} rows done")

print(f"  Reasoning generation complete in {time.time() - t_start:.1f}s")


# ── Assemble the output dataframe
output_df = pd.DataFrame({
    'id':                    test_ids,
    
    # Context columns (useful for demo / inspection)
    'driver':                kaggle_test_raw['Driver'].values,
    'race':                  kaggle_test_raw['Race'].values,
    'year':                  kaggle_test_raw['Year'].values,
    'lap':                   kaggle_test_raw['LapNumber'].values,
    'compound':              kaggle_test_raw['Compound'].values,
    'tyre_life':             kaggle_test_raw['TyreLife'].values,
    
    # Required columns
    'final_call':            decisions,
    'pit_prob':              np.round(test_probs, 4),
    'degradation_severity':  np.round(test_concepts[:, 0], 4),
    'pace_decay_rate':       np.round(test_concepts[:, 1], 4),
    'strategic_window':      np.round(test_concepts[:, 2], 4),
    'track_position_risk':   np.round(test_concepts[:, 3], 4),
    'undercut_pressure':     np.round(test_concepts[:, 4], 4),
    'endgame_proximity':     np.round(test_concepts[:, 5], 4),
    'reasoning':             reasoning_list,
})

print(f"\n  Output shape: {output_df.shape}")
print(f"  Columns: {list(output_df.columns)}")


# ── Save to CSV
output_path = OUTPUTS_DIR / 'inference_with_reasoning.csv'
output_df.to_csv(output_path, index=False)
print(f"\n  Saved to: {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")


# ── Show a few sample rows
print("\n" + "=" * 70)
print("SAMPLE OUTPUT ROWS")
print("=" * 70)

# Pick a confident pit, a confident no-pit, and a borderline
high_idx = int(np.argmax(test_probs))
low_idx = int(np.argmin(test_probs))
mid_idx = int(np.argmin(np.abs(test_probs - 0.5)))

for label, idx in [("HIGH PIT PROB", high_idx),
                   ("LOW PIT PROB", low_idx),
                   ("BORDERLINE", mid_idx)]:
    row = output_df.iloc[idx]
    print(f"\n  --- {label} (id={int(row['id'])}) ---")
    print(f"  Driver: {row['driver']}, {row['race']} {int(row['year'])}, "
          f"Lap {int(row['lap'])}, {row['compound']} (TyreLife {int(row['tyre_life'])})")
    print(f"  Final call: {row['final_call']} (probability {row['pit_prob']:.3f})")
    print(f"  Concept scores:")
    for cn in CONCEPT_NAMES:
        print(f"    {cn:<24s} {row[cn]:.4f}")
    print(f"  Reasoning: \"{row['reasoning']}\"")

print("\n" + "=" * 70)
print(f"  Total PIT predictions:    {(decisions == 'PIT').sum():,} ({(decisions == 'PIT').mean()*100:.1f}%)")
print(f"  Total NO_PIT predictions: {(decisions == 'NO_PIT').sum():,} ({(decisions == 'NO_PIT').mean()*100:.1f}%)")
print("=" * 70)

LOADING JOINT λ=20 MODEL

  Lambda used:      20.0
  Best val AUC:     0.8918233999249967
  Total params:     717

  Decision block weights:
    degradation_severity      +3.9078
    pace_decay_rate           -5.2246
    strategic_window          +0.1456
    track_position_risk       +0.1301
    undercut_pressure         +0.3570
    endgame_proximity         -36.3405
    bias                      +1.4217

LOADING KAGGLE TEST DATA
  Test rows: 188,165

RUNNING INFERENCE
  Processed 50,000 / 188,165
  Processed 100,000 / 188,165
  Processed 150,000 / 188,165
  Processed 188,165 / 188,165

  Inference complete in 0.1s

BUILDING OUTPUT TABLE WITH REASONING
  Generating reasoning for 188,165 rows...
    20,000 rows done
    40,000 rows done
    60,000 rows done
    80,000 rows done
    100,000 rows done
    120,000 rows done
    140,000 rows done
    160,000 rows done
    180,000 rows done
  Reasoning generation complete in 1.7s

  Output shape: (188165, 16)
  Columns: ['id', 'driver', 'rac

In [5]:
def find_scenario_candidates_v2(scenario, n=5):
    """Tightened scenario selection. Fixes:
    1. Late-race scenarios now require Lap > 30 to avoid corrupted RaceProgress
    2. Defend-position now requires undercut_pressure LOW (otherwise reasoning is incoherent)
    3. Strategic-window now requires undercut_pressure LOW (otherwise undercut dominates the narrative)
    """
    rp = kaggle_test_raw['RaceProgress'].values
    lap = kaggle_test_raw['LapNumber'].values
    tyre_life = kaggle_test_raw['TyreLife'].values
    stint = kaggle_test_raw['Stint'].values
    race = kaggle_test_raw['Race'].values
    pos = kaggle_test_raw['Position'].values
    
    is_real_race = ~np.array(['Testing' in str(r) or 'Pre-Season' in str(r) for r in race])
    c = test_concepts
    
    if scenario == 'undercut_response':
        mask = (
            is_real_race & (lap >= 15) &
            (rp > 0.25) & (rp < 0.70) &
            (c[:, 4] > 0.9) &
            (c[:, 0] < 0.4) &              # NOT dominated by degradation
            (tyre_life > 8) & (tyre_life < 30) &
            (test_probs > 0.7)
        )
    
    elif scenario == 'degradation_stop':
        mask = (
            is_real_race & (lap >= 20) &
            (rp > 0.25) & (rp < 0.75) &
            (c[:, 0] > 0.9) &              # very high degradation
            (c[:, 4] < 0.3) &              # NOT confounded by undercut
            (tyre_life > 25) &
            (test_probs > 0.6)
        )
    
    elif scenario == 'late_race_hold':
        # CRITICAL: require Lap > 30 to avoid the lap-1/RaceProgress=1.0 corruption
        mask = (
            is_real_race &
            (lap >= 30) &                  # genuine late-race lap number
            (rp > 0.85) &
            (c[:, 5] > 0.6) &
            (tyre_life > 10) &             # rule out fresh-tyre nonsense
            (test_probs < 0.3)
        )
    
    elif scenario == 'defend_position':
        # Track-position-risk dominates, NO undercut confound
        mask = (
            is_real_race & (lap >= 15) &
            (rp > 0.3) & (rp < 0.7) &
            (c[:, 3] > 0.7) &
            (c[:, 4] < 0.3) &              # NOT confounded by undercut
            (pos <= 5) &                   # genuinely strong position
            (test_probs < 0.5)
        )
    
    elif scenario == 'strategic_window_open':
        # Window dominates, NO undercut confound
        mask = (
            is_real_race & (lap >= 15) &
            (rp > 0.3) & (rp < 0.7) &
            (c[:, 2] > 0.8) &
            (c[:, 4] < 0.3) &              # NOT confounded by undercut
            (test_probs > 0.6)
        )
    
    elif scenario == 'borderline':
        mask = (
            is_real_race & (lap >= 15) &
            (rp > 0.3) & (rp < 0.7) &
            (np.abs(test_probs - 0.5) < 0.05) &
            (tyre_life > 8)
        )
    
    candidates = np.where(mask)[0]
    if len(candidates) == 0:
        return []
    
    signature_concept = {
        'undercut_response': 4, 'degradation_stop': 0, 'late_race_hold': 5,
        'defend_position': 3, 'strategic_window_open': 2, 'borderline': None,
    }[scenario]
    
    if signature_concept is not None:
        candidates = candidates[np.argsort(c[candidates, signature_concept])[::-1]]
    
    return candidates[:n].tolist()

In [9]:
def coherence_filter(concepts, raw_row):
    """Return (filtered_concepts, suppressed_list).
    
    filtered_concepts: copy with incoherent concepts zeroed for narration.
    suppressed_list: names of concepts that were suppressed (for audit).
    """
    c = concepts.copy()
    suppressed = []
    
    race_progress = float(raw_row['RaceProgress'])
    tyre_life = float(raw_row['TyreLife'])
    lap = int(raw_row['LapNumber'])
    stint = int(raw_row['Stint'])
    race_name = str(raw_row['Race'])
    
    # 1. endgame_proximity should only fire late in a race
    if c[5] > 0.4 and race_progress < 0.6:
        c[5] = 0.0
        suppressed.append('endgame_proximity')
    
    # 2. degradation_severity shouldn't be high on near-new tyres
    if c[0] > 0.5 and tyre_life <= 3:
        c[0] = 0.0
        suppressed.append('degradation_severity')
    
    # 3. pace_decay_rate is implausible on the first lap of a stint
    if c[1] > 0.3 and tyre_life <= 1:
        c[1] = 0.0
        suppressed.append('pace_decay_rate')
    
    # 4. strategic_window and undercut_pressure are implausible on laps 1-2
    if lap <= 2:
        if c[2] > 0.2:
            c[2] = 0.0
            suppressed.append('strategic_window')
        if c[4] > 0.2:
            c[4] = 0.0
            suppressed.append('undercut_pressure')
    
    # 5. Non-race events shouldn't be narrated as strategy
    if 'Testing' in race_name or 'Pre-Season' in race_name:
        c[:] = 0.0
        suppressed.append('ALL (non-race event)')
    
    return c, suppressed

In [10]:
# ── Run v2 selection across all scenarios

print("=" * 70)
print("CURATED DEMO SCENARIO CANDIDATES (v2 — tightened selection)")
print("=" * 70)

scenarios = [
    ('undercut_response',      "Undercut response — competitor pitted, must react"),
    ('degradation_stop',       "Degradation stop — tyres worn out, time to change"),
    ('late_race_hold',         "Late-race hold — too close to the end to pit"),
    ('defend_position',        "Defend position — protect track position, stay out"),
    ('strategic_window_open',  "Strategic window — optimal pit timing, commit now"),
    ('borderline',             "Borderline — genuine 50/50 judgment call"),
]

for scenario_key, scenario_desc in scenarios:
    candidates = find_scenario_candidates_v2(scenario_key, n=5)
    print(f"\n{'─' * 70}")
    print(f"SCENARIO: {scenario_desc}")
    print(f"{'─' * 70}")
    
    if len(candidates) == 0:
        print("  No coherent candidates found. Selection criteria may need loosening.")
        continue
    
    print(f"  {len(candidates)} candidate(s) found.\n")
    
    for rank, idx in enumerate(candidates, 1):
        row = kaggle_test_raw.iloc[idx]
        concepts = test_concepts[idx]
        prob = test_probs[idx]
        decision = "PIT" if prob >= 0.5 else "NO_PIT"
        reasoning = build_strategic_reasoning(
            test_concepts[idx], contributions_all[idx], decision, test_ids[idx],
            raw_row=row
        )
        
        print(f"  [Candidate {rank}] id={int(row['id'])}")
        print(f"    {row['Driver']}, {row['Race']} {int(row['Year'])}, "
              f"Lap {int(row['LapNumber'])} (RaceProgress {row['RaceProgress']:.2f})")
        print(f"    {row['Compound']} TyreLife={int(row['TyreLife'])}, "
              f"P{int(row['Position'])}, Stint {int(row['Stint'])}")
        print(f"    Decision: {decision} ({prob:.1%})")
        print(f"    Concepts: deg={concepts[0]:.2f} pace={concepts[1]:.2f} "
              f"window={concepts[2]:.2f} posrisk={concepts[3]:.2f} "
              f"undercut={concepts[4]:.2f} endgame={concepts[5]:.2f}")
        print(f"    Reasoning: \"{reasoning}\"")
        print()

CURATED DEMO SCENARIO CANDIDATES (v2 — tightened selection)

──────────────────────────────────────────────────────────────────────
SCENARIO: Undercut response — competitor pitted, must react
──────────────────────────────────────────────────────────────────────
  5 candidate(s) found.

  [Candidate 1] id=557736
    KUB, Australian Grand Prix 2025, Lap 34 (RaceProgress 0.45)
    HARD TyreLife=11, P7, Stint 5
    Decision: PIT (90.0%)
    Concepts: deg=0.08 pace=0.00 window=0.34 posrisk=1.00 undercut=1.00 endgame=0.00
    Reasoning: "a rival has pitted and is poised to undercut. Failing to respond this lap likely hands them the position once stops cycle through, so a stop is justified now."

  [Candidate 2] id=587369
    DRA, Japanese Grand Prix 2024, Lap 31 (RaceProgress 0.44)
    HARD TyreLife=18, P10, Stint 4
    Decision: PIT (93.1%)
    Concepts: deg=0.32 pace=0.02 window=0.24 posrisk=1.00 undercut=1.00 endgame=0.01
    Reasoning: "the tyres are picking up early wear. Degradation i